Packages that are being used in this notebook:

In [1]:
!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch

Cloning into 'reasoning-from-scratch'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (236/236), done.
remote: Total 269 (delta 45), reused 160 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 1.79 MiB | 19.73 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [2]:
import sys
import os
import torch

# Add the cloned repository directory to Python's system path
repo_path = os.path.abspath("reasoning-from-scratch")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer = load_model_and_tokenizer(
    which_model="reasoning",
    device=device,
    use_compile=False
)

qwen3-0.6B-reasoning.pth: 100% (1433 MiB / 1433 MiB)
tokenizer-reasoning.json: 100% (10 MiB / 10 MiB)


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=256, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

<think>
Okay, let's see. The problem says that half of (3x minus 9) is equal to x plus 37. I need to find the value of x. Hmm, let me write that down to make sure I have it right.

So, the equation given is: (1/2)(3x - 9) = x + 37. Right? And I need to solve for x.

Alright, to solve this equation, I think the first step is to eliminate the fraction. Since it's half of (3x - 9), maybe I can multiply both sides by 2 to get rid of the denominator. Let me try that.

Multiplying both sides by 2 gives me: 2 * (1/2)(3x - 9) = 2*(x + 37). Simplifying the left side, the 2 and the 1/2 cancel out, so it's just (3x - 9). On the right side, 2 times x is 2x and 2 times 37 is 74. So now the equation is 3x - 9 = 2x + 74.

Now, I need to solve for x

In [4]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [5]:
math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [6]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

In [7]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for step in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [8]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=256,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

<think>
Okay, let's see. The problem says that half of (3x minus 9) is equal to x plus 37. I need to find the value of x. Hmm, let me write that down to make sure I have it right.

So, the equation given is: (1/2)(3x - 9) = x + 37. Right? And I need to solve for x.

Alright, to solve this equation, I think the first step is to eliminate the fraction. Since it's half of (3x - 9), maybe I can multiply both sides by 2 to get rid of the denominator. Let me try that.

Multiplying both sides by 2 gives me: 2 * (1/2)(3x - 9) = 2*(x + 37). Simplifying the left side, the 2 and the 1/2 cancel out, so it's just (3x - 9). On the right side, 2 times x is 2x and 2 times 37 is 74. So now the equation is 3x - 9 = 2x + 74.

Now, I need to solve for x


In [9]:
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=256,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

<think>
Okay, let's see. The problem says that half the value of 3x minus 9 is equal to x plus 37. I need to find the value of x. Hmm, let me write that down to visualize it better.

So, the equation is: (1/2)(3x - 9) = x + 37. Alright, that makes sense. Now, I need to solve for x. Let me think about how to approach this.

First, maybe I should multiply both sides by 2 to get rid of the fraction. That way, I can simplify the equation. Let me try that. If I multiply both sides by 2, the left side becomes 3x - 9, and the right side becomes 2x + 74. Wait, let me check that again. If I multiply (1/2)(3x - 9) by 2, it's indeed 3x - 9. And the right side, x + 37 multiplied by 2 is 2x + 74. Yep, that seems right.

So now the equation is 3x - 9 = 2x + 74. Now, to solve for x


In [10]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

In [11]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [12]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



In [13]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.], device='cuda:0')


In [14]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659], device='cuda:0')


In [15]:
## Chapter 5

@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

In [16]:
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer,
                   prompt=prompt,
                   answer=answer_text,
                   device=device)
print(avg_logprob_val)

-0.625


In [17]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

-165.0


In [18]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-38.9937, device='cuda:0', grad_fn=<SumBackward0>)


In [19]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-38.9937, device='cuda:0', grad_fn=<SumBackward0>)


In [20]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -65.1005

Answer:  The correct answer is \boxed{83}
Logprob: -71.6610

Answer:  The final answer is 83
Logprob: -69.1485

Answer:  We get \boxed{38}
Logprob: -69.5194



In [21]:
logps = torch.stack(rollout_logps)
print(logps)

tensor([-65.1005, -71.6610, -69.1485, -69.5194], device='cuda:0',
       grad_fn=<StackBackward0>)


In [22]:
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

tensor(-0.4127, device='cuda:0', grad_fn=<NegBackward0>)


In [23]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])

        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [24]:
torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., device='cuda:0', grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 256,
              'reward': 0.0,
              'text': '<think>\n'
                      "Okay, let's see. Sam is working for 20 days, right? And "
                      "each day he works earns him $60. But if he doesn't "
                      'work, he gets $30 less. At the end of the 20 days, he '
                      'earned $660. The question is asking how many days he '
                      "didn't work. Hmm, so I need to figure out the total "
                      'days he worked and then subtract that from 20 to find '
                      "out how many days he didn't work.\n"
                      '\n'
                      'First, let me confirm the total earnings. He earned '
                      '$660 in total. Since he works 20 days, and each day he '
                      'earns $60, maybe

In [25]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimizer
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=lr,
    fused=True
)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"])
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

In [26]:
import gc
import torch



model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=7,
    num_rollouts=2,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

[Step 1/7] loss=-0.0000 reward_avg=0.000 avg_resp_len=512.0
[Step 2/7] loss=-0.0000 reward_avg=0.000 avg_resp_len=512.0
[Step 3/7] loss=-0.0000 reward_avg=1.000 avg_resp_len=375.5
[Step 4/7] loss=-13.0398 reward_avg=0.500 avg_resp_len=413.0
[Step 5/7] loss=-0.0000 reward_avg=0.000 avg_resp_len=512.0
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth
[Step 6/7] loss=-0.0000 reward_avg=0.000 avg_resp_len=512.0
[Step 7/7] loss=-0.0000 reward_avg=0.000 avg_resp_len=512.0


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [ ]:
import torch

# Define a few math test problems
test_problems = [
    "Let $a$, $b$, $c$ be the roots of $x^3 - 7x^2 + 5x + 2 = 0$. What is the value of $a^3 + b^3 + c^3$?",
    "What is the remainder when $2^{2026}$ is divided by $1000$?",
    "What is the sum of all positive divisors of $2026^2$?",
    "How many integer pairs $(x, y)$ satisfy $x^2 + y^2 \\le 100$?",
    "How many positive integers $n \\le 1000$ are there such that $n^2 + n + 1$ is divisible by $7$?",
]

# Switch model to evaluation mode
model.eval()

for i, prob in enumerate(test_problems, 1):
    print(f"=== Test Problem {i} ===")
    print(f"Question: {prob}\n")

    # Render the input with the system template
    test_prompt = render_prompt(prob)

    # Generate a response using our sampling helper
    torch.manual_seed(42)  # Set seed for reproducibility
    _, _, generated_response = sample_response(
        model=model,
        tokenizer=tokenizer,
        prompt=test_prompt,
        device=device,
        max_new_tokens=2048,
        temperature=0.8,
        top_p=0.9
    )

    print("Generated Output:")
    print(generated_response)
    print("=" * 30 + "\n")

=== Test Problem 1 ===
Question: Let $a$, $b$, $c$ be the roots of $x^3 - 7x^2 + 5x + 2 = 0$. What is the value of $a^3 + b^3 + c^3$?

Generated Output:
<think>
Okay, let's see. I need to find the value of $a^3 + b^3 + c^3$ where $a$, $b$, and $c$ are the roots of the cubic equation $x^3 - 7x^2 + 5x + 2 = 0$. Hmm, I remember there's something called Vieta's formulas that relate the coefficients of a polynomial to sums and products of its roots. Let me recall... Vieta's formulas for a cubic equation $x^3 + px^2 + qx + r = 0$ tell us that the sum of the roots $a + b + c = -p$, the sum of the products $ab + bc + ca = q$, and the product $abc = -r$. Wait, so in this case, the given equation is $x^3 - 7x^2 + 5x + 2 = 0$, which can be rewritten as $x^3 + (-7)x^2 + 5x + 2 = 0$. Comparing with the general form, that means $p = -7$, $q = 5$, and $r = 2$. 

So applying Vieta's formulas, the sum of the roots $a + b + c = -p = -(-7) = 7$. The sum of the products $ab + bc + ca = q = 5$. And the pro